In [1]:
# Setup imports and paths
import os
import sys
from pathlib import Path
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import transforms
from tqdm.notebook import tqdm

# Add repo root to path
ROOT = Path('..').resolve() / '..'  # notebooks/attribution -> repo root
ROOT = Path('.') if not (ROOT / 'src').exists() else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.factory import get_model
from src.datasets.isic import ISICDataset
from src.xai.attribution_unified import (
    IntegratedGradientsUnified,
    GradCAMUnified,
    RISEUnified,
    OcclusionUnified
)

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Repo root: {ROOT.absolute()}')

Using device: cpu
PyTorch version: 2.9.1+cpu
Repo root: D:\git projects\certified-attribution-medical-imaging\notebooks\..


In [ ]:
# Configuration
# Try multiple checkpoint locations
CHECKPOINT_OPTIONS = [
    ROOT / 'checkpoints' / 'isic',                              # Repo root (preferred)
    ROOT / 'notebooks' / 'training' / 'checkpoints' / 'isic',  # Training notebook location
    Path('.') / '..' / 'training' / 'checkpoints' / 'isic'     # Relative to current notebook
]

CHECKPOINT_DIR = None
for path in CHECKPOINT_OPTIONS:
    if path.exists():
        CHECKPOINT_DIR = path
        print(f'✓ Found checkpoints at: {CHECKPOINT_DIR.absolute()}')
        break

if CHECKPOINT_DIR is None:
    CHECKPOINT_DIR = CHECKPOINT_OPTIONS[0]  # Use default if none exist
    print(f'⚠ Using default checkpoint path (will create if needed): {CHECKPOINT_DIR.absolute()}')

DATA_ROOT = ROOT / 'data' / 'raw' / 'isic'
OUTPUT_DIR = ROOT / 'outputs' / 'attributions_viz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Attribution parameters
NUM_SAMPLES = 8  # Number of test images to analyze
IMG_SIZE = 224

# Attribution method parameters
ATTRIBUTION_METHODS = {
    'IntegratedGradients': {'num_steps': 50},
    'GradCAM': {},
    'RISE': {'num_samples': 500, 'mask_size': 7},
    'Occlusion': {'patch_size': 16, 'stride': 8}
}

print(f'Checkpoint directory: {CHECKPOINT_DIR.absolute()}')
print(f'Data directory: {DATA_ROOT.absolute()}')
print(f'Output directory: {OUTPUT_DIR.absolute()}')


Checkpoint directory: D:\git projects\certified-attribution-medical-imaging\notebooks\..\checkpoints\isic
Data directory: D:\git projects\certified-attribution-medical-imaging\notebooks\..\data\raw\isic
Output directory: D:\git projects\certified-attribution-medical-imaging\notebooks\..\outputs\attributions_viz


In [6]:
# Discover available trained models
print(f'Searching for models in: {CHECKPOINT_DIR.absolute()}')
print(f'Directory exists: {CHECKPOINT_DIR.exists()}')

available_models = []

if CHECKPOINT_DIR.exists():
    print(f'Contents of {CHECKPOINT_DIR}:')
    for item in CHECKPOINT_DIR.iterdir():
        print(f'  - {item.name} (is_dir: {item.is_dir()})')
    
    for model_dir in CHECKPOINT_DIR.iterdir():
        if model_dir.is_dir():
            best_ckpt = model_dir / 'best_model.pt'
            final_ckpt = model_dir / 'final_model.pt'
            
            print(f'\n  Checking {model_dir.name}:')
            print(f'    best_model.pt exists: {best_ckpt.exists()}')
            print(f'    final_model.pt exists: {final_ckpt.exists()}')
            
            if best_ckpt.exists() or final_ckpt.exists():
                checkpoint_path = best_ckpt if best_ckpt.exists() else final_ckpt
                available_models.append({
                    'name': model_dir.name,
                    'checkpoint': checkpoint_path,
                    'type': 'best' if best_ckpt.exists() else 'final'
                })
else:
    print(f'⚠ Directory does not exist: {CHECKPOINT_DIR.absolute()}')

print(f'\nFound {len(available_models)} trained models:')
print('='*60)
for i, model_info in enumerate(available_models, 1):
    print(f"{i}. {model_info['name']:<20} ({model_info['type']} checkpoint)")
    print(f"   Path: {model_info['checkpoint']}")
print('='*60)

if not available_models:
    print('\n⚠ No trained models found!')
    print(f'Expected path: {CHECKPOINT_DIR.absolute()}')
    print(f'Please check:')
    print(f'  1. Run train_isic_models.ipynb first')
    print(f'  2. Check if checkpoints exist in the expected directory')


Searching for models in: D:\git projects\certified-attribution-medical-imaging\notebooks\..\checkpoints\isic
Directory exists: False
⚠ Directory does not exist: D:\git projects\certified-attribution-medical-imaging\notebooks\..\checkpoints\isic

Found 0 trained models:

⚠ No trained models found!
Expected path: D:\git projects\certified-attribution-medical-imaging\notebooks\..\checkpoints\isic
Please check:
  1. Run train_isic_models.ipynb first
  2. Check if checkpoints exist in the expected directory


In [ ]:
# Load validation dataset
print('Loading ISIC validation dataset...')

# Transform for visualization (no normalization yet)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

val_dataset = ISICDataset(
    str(DATA_ROOT),
    split='val',
    transform=val_transform,
    target_size=(IMG_SIZE, IMG_SIZE)
)

print(f'✓ Loaded {len(val_dataset)} validation images')

# Get class names
if hasattr(val_dataset, 'classes'):
    class_names = val_dataset.classes
elif hasattr(val_dataset, 'label_map'):
    class_names = list(val_dataset.label_map.values())
else:
    class_names = [f'Class {i}' for i in range(8)]

print(f'Classes: {class_names}')

In [ ]:
# Select random samples for analysis
np.random.seed(42)
sample_indices = np.random.choice(len(val_dataset), NUM_SAMPLES, replace=False)

print(f'Selected {NUM_SAMPLES} random samples for attribution analysis:')
print('='*60)
for i, idx in enumerate(sample_indices, 1):
    sample = val_dataset[idx]
    label = sample['label']
    class_name = class_names[label] if label < len(class_names) else f'Class {label}'
    print(f'{i}. Sample {idx}: {class_name}')
print('='*60)

In [ ]:
# Helper function to get target layer for Grad-CAM
def get_target_layer(model, model_name):
    """Get the last convolutional layer for Grad-CAM."""
    if 'resnet' in model_name.lower():
        return model.layer4[-1].conv2
    elif 'densenet' in model_name.lower():
        return model.features.denseblock4
    elif 'efficientnet' in model_name.lower():
        return model.features[-1][0]
    elif 'mobilenet' in model_name.lower():
        return model.features[-1][0]
    else:
        # Fallback: try to find last Conv2d
        for name, module in reversed(list(model.named_modules())):
            if isinstance(module, torch.nn.Conv2d):
                return module
        raise ValueError(f'Could not find target layer for {model_name}')

# Helper function to apply ImageNet normalization
def normalize_image(image):
    """Apply ImageNet normalization to image tensor."""
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    return normalize(image)

# Helper function to overlay heatmap on image
def overlay_heatmap(image, heatmap, alpha=0.5, colormap='jet'):
    """
    Overlay heatmap on image.
    
    Args:
        image: numpy array [H, W, 3] in range [0, 1]
        heatmap: numpy array [H, W] in range [0, 1]
        alpha: transparency of heatmap
        colormap: matplotlib colormap name
    
    Returns:
        overlaid image [H, W, 3]
    """
    import matplotlib.cm as cm
    
    # Resize heatmap to match image size if needed
    if heatmap.shape != image.shape[:2]:
        from scipy.ndimage import zoom
        zoom_factor = (image.shape[0] / heatmap.shape[0], image.shape[1] / heatmap.shape[1])
        heatmap = zoom(heatmap, zoom_factor, order=1)
    
    # Apply colormap
    cmap = cm.get_cmap(colormap)
    heatmap_colored = cmap(heatmap)[:, :, :3]  # Remove alpha channel
    
    # Blend
    overlaid = (1 - alpha) * image + alpha * heatmap_colored
    return np.clip(overlaid, 0, 1)

print('Helper functions defined ✓')

In [ ]:
# Generate attributions for all models and methods
attribution_results = {}

for model_info in tqdm(available_models, desc='Models'):
    model_name = model_info['name']
    checkpoint_path = model_info['checkpoint']
    
    print(f'\n{"="*60}')
    print(f'Processing: {model_name}')
    print(f'{"="*60}')
    
    # Load model
    try:
        # Get number of classes from dataset
        num_classes = len(class_names)
        
        # Build model architecture
        model, cfg = get_model(model_name, num_classes=num_classes, pretrained=False, device=DEVICE)
        
        # Load weights
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f'✓ Model loaded successfully')
        
    except Exception as e:
        print(f'✗ Failed to load model: {e}')
        continue
    
    # Initialize attribution methods
    try:
        target_layer = get_target_layer(model, model_name)
        
        attribution_methods = {
            'IntegratedGradients': IntegratedGradientsUnified(model, DEVICE),
            'GradCAM': GradCAMUnified(model, target_layer, DEVICE),
            'RISE': RISEUnified(model, DEVICE),
            'Occlusion': OcclusionUnified(model, DEVICE)
        }
        print(f'✓ Attribution methods initialized')
        
    except Exception as e:
        print(f'✗ Failed to initialize attribution methods: {e}')
        continue
    
    # Store results for this model
    attribution_results[model_name] = {
        'predictions': [],
        'attributions': {method: [] for method in ATTRIBUTION_METHODS.keys()}
    }
    
    # Process each sample
    for sample_idx in tqdm(sample_indices, desc=f'Samples ({model_name})', leave=False):
        sample = val_dataset[sample_idx]
        image = sample['image']  # [C, H, W] unnormalized
        true_label = sample['label']
        
        # Get prediction
        with torch.no_grad():
            normalized_image = normalize_image(image).unsqueeze(0).to(DEVICE)
            output = model(normalized_image)
            pred_label = output.argmax(dim=1).item()
            pred_prob = torch.softmax(output, dim=1)[0, pred_label].item()
        
        attribution_results[model_name]['predictions'].append({
            'sample_idx': sample_idx,
            'true_label': true_label,
            'pred_label': pred_label,
            'pred_prob': pred_prob,
            'correct': true_label == pred_label
        })
        
        # Generate attributions for each method
        for method_name, method_params in ATTRIBUTION_METHODS.items():
            try:
                method = attribution_methods[method_name]
                
                # Apply normalization before attribution
                normalized_image_single = normalize_image(image)
                
                # Generate attribution
                heatmap = method.attribute(
                    normalized_image_single,
                    target_class=pred_label,
                    **method_params
                )
                
                attribution_results[model_name]['attributions'][method_name].append(heatmap)
                
            except Exception as e:
                print(f'✗ {method_name} failed for sample {sample_idx}: {e}')
                attribution_results[model_name]['attributions'][method_name].append(None)
    
    print(f'✓ Completed {model_name}')

print(f'\n{"="*60}')
print('Attribution generation complete!')
print(f'{"="*60}')

In [ ]:
# Visualize attributions for a single sample across all models and methods
def visualize_sample_attributions(sample_idx_in_list, figsize=(20, 12)):
    """
    Visualize all attributions for a single sample.
    
    Creates a grid showing:
    - Rows: Different models
    - Columns: Original image + Attribution methods
    """
    sample_idx = sample_indices[sample_idx_in_list]
    sample = val_dataset[sample_idx]
    image = sample['image'].permute(1, 2, 0).numpy()  # [H, W, C]
    true_label = sample['label']
    true_class = class_names[true_label] if true_label < len(class_names) else f'Class {true_label}'
    
    num_models = len(available_models)
    num_cols = 1 + len(ATTRIBUTION_METHODS)  # Original + methods
    
    fig, axes = plt.subplots(num_models, num_cols, figsize=figsize)
    if num_models == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f'Sample {sample_idx} - True Class: {true_class}', 
                fontsize=16, fontweight='bold', y=0.995)
    
    for row_idx, model_info in enumerate(available_models):
        model_name = model_info['name']
        results = attribution_results[model_name]
        pred_info = results['predictions'][sample_idx_in_list]
        
        pred_label = pred_info['pred_label']
        pred_prob = pred_info['pred_prob']
        pred_class = class_names[pred_label] if pred_label < len(class_names) else f'Class {pred_label}'
        is_correct = pred_info['correct']
        
        # Column 0: Original image with prediction
        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].axis('off')
        title_color = 'green' if is_correct else 'red'
        axes[row_idx, 0].set_title(
            f'{model_name}\nPred: {pred_class} ({pred_prob:.2%})',
            fontsize=10, fontweight='bold', color=title_color
        )
        
        # Remaining columns: Attribution methods
        for col_idx, method_name in enumerate(ATTRIBUTION_METHODS.keys(), start=1):
            heatmap = results['attributions'][method_name][sample_idx_in_list]
            
            if heatmap is not None:
                overlaid = overlay_heatmap(image, heatmap, alpha=0.5)
                axes[row_idx, col_idx].imshow(overlaid)
            else:
                axes[row_idx, col_idx].text(0.5, 0.5, 'Failed', 
                                           ha='center', va='center',
                                           transform=axes[row_idx, col_idx].transAxes)
            
            axes[row_idx, col_idx].axis('off')
            if row_idx == 0:
                axes[row_idx, col_idx].set_title(method_name, fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    save_path = OUTPUT_DIR / f'sample_{sample_idx}_all_attributions.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {save_path}')
    
    plt.show()

# Visualize first sample
if attribution_results:
    visualize_sample_attributions(0)

In [ ]:
# Visualize all samples for one model
def visualize_model_attributions(model_name, figsize=(20, 16)):
    """
    Visualize all samples for a single model.
    
    Creates a grid showing:
    - Rows: Different samples
    - Columns: Original image + Attribution methods
    """
    if model_name not in attribution_results:
        print(f'Model {model_name} not found in results')
        return
    
    results = attribution_results[model_name]
    num_samples = len(sample_indices)
    num_cols = 1 + len(ATTRIBUTION_METHODS)
    
    fig, axes = plt.subplots(num_samples, num_cols, figsize=figsize)
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f'Model: {model_name} - Attribution Analysis', 
                fontsize=16, fontweight='bold', y=0.995)
    
    for row_idx, sample_idx_in_list in enumerate(range(num_samples)):
        sample_idx = sample_indices[sample_idx_in_list]
        sample = val_dataset[sample_idx]
        image = sample['image'].permute(1, 2, 0).numpy()
        true_label = sample['label']
        true_class = class_names[true_label] if true_label < len(class_names) else f'Class {true_label}'
        
        pred_info = results['predictions'][sample_idx_in_list]
        pred_label = pred_info['pred_label']
        pred_prob = pred_info['pred_prob']
        pred_class = class_names[pred_label] if pred_label < len(class_names) else f'Class {pred_label}'
        is_correct = pred_info['correct']
        
        # Column 0: Original image
        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].axis('off')
        title_color = 'green' if is_correct else 'red'
        axes[row_idx, 0].set_title(
            f'Sample {sample_idx}\nTrue: {true_class}\nPred: {pred_class} ({pred_prob:.2%})',
            fontsize=9, fontweight='bold', color=title_color
        )
        
        # Attribution methods
        for col_idx, method_name in enumerate(ATTRIBUTION_METHODS.keys(), start=1):
            heatmap = results['attributions'][method_name][sample_idx_in_list]
            
            if heatmap is not None:
                overlaid = overlay_heatmap(image, heatmap, alpha=0.5)
                axes[row_idx, col_idx].imshow(overlaid)
            else:
                axes[row_idx, col_idx].text(0.5, 0.5, 'Failed',
                                           ha='center', va='center',
                                           transform=axes[row_idx, col_idx].transAxes)
            
            axes[row_idx, col_idx].axis('off')
            if row_idx == 0:
                axes[row_idx, col_idx].set_title(method_name, fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    
    # Save
    save_path = OUTPUT_DIR / f'{model_name}_all_samples.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {save_path}')
    
    plt.show()

# Visualize first available model
if attribution_results:
    first_model = list(attribution_results.keys())[0]
    visualize_model_attributions(first_model)

In [ ]:
# Compare attribution methods for a single sample and model
def compare_attribution_methods(model_name, sample_idx_in_list, figsize=(16, 4)):
    """
    Compare attribution methods side-by-side for one model and sample.
    Shows original, heatmaps, and overlays.
    """
    if model_name not in attribution_results:
        print(f'Model {model_name} not found')
        return
    
    sample_idx = sample_indices[sample_idx_in_list]
    sample = val_dataset[sample_idx]
    image = sample['image'].permute(1, 2, 0).numpy()
    true_label = sample['label']
    true_class = class_names[true_label] if true_label < len(class_names) else f'Class {true_label}'
    
    results = attribution_results[model_name]
    pred_info = results['predictions'][sample_idx_in_list]
    pred_class = class_names[pred_info['pred_label']] if pred_info['pred_label'] < len(class_names) else f"Class {pred_info['pred_label']}"
    
    num_methods = len(ATTRIBUTION_METHODS)
    fig, axes = plt.subplots(2, num_methods + 1, figsize=figsize)
    
    fig.suptitle(
        f'{model_name} - Sample {sample_idx}\nTrue: {true_class}, Pred: {pred_class} ({pred_info["pred_prob"]:.2%})',
        fontsize=14, fontweight='bold'
    )
    
    # Column 0: Original image
    axes[0, 0].imshow(image)
    axes[0, 0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')  # Empty bottom cell
    
    # Other columns: Attribution methods
    for col_idx, method_name in enumerate(ATTRIBUTION_METHODS.keys(), start=1):
        heatmap = results['attributions'][method_name][sample_idx_in_list]
        
        if heatmap is not None:
            # Row 0: Heatmap only
            axes[0, col_idx].imshow(heatmap, cmap='jet')
            axes[0, col_idx].set_title(f'{method_name}\n(Heatmap)', fontsize=9)
            axes[0, col_idx].axis('off')
            
            # Row 1: Overlay
            overlaid = overlay_heatmap(image, heatmap, alpha=0.5)
            axes[1, col_idx].imshow(overlaid)
            axes[1, col_idx].set_title('Overlay', fontsize=9)
            axes[1, col_idx].axis('off')
        else:
            for row in [0, 1]:
                axes[row, col_idx].text(0.5, 0.5, 'Failed',
                                       ha='center', va='center',
                                       transform=axes[row, col_idx].transAxes)
                axes[row, col_idx].axis('off')
    
    plt.tight_layout()
    
    # Save
    save_path = OUTPUT_DIR / f'{model_name}_sample{sample_idx}_comparison.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'✓ Saved: {save_path}')
    
    plt.show()

# Compare for first model and sample
if attribution_results:
    first_model = list(attribution_results.keys())[0]
    compare_attribution_methods(first_model, 0)

In [ ]:
# Generate all visualizations systematically
print('Generating comprehensive visualizations...')
print('='*60)

# 1. For each sample: compare all models
for i in range(len(sample_indices)):
    print(f'\nVisualizing sample {i+1}/{len(sample_indices)} across all models...')
    visualize_sample_attributions(i, figsize=(20, 4 * len(available_models)))

# 2. For each model: compare all samples
for model_name in attribution_results.keys():
    print(f'\nVisualizing all samples for {model_name}...')
    visualize_model_attributions(model_name, figsize=(20, 3 * len(sample_indices)))

# 3. Detailed comparisons for first 3 samples of each model
for model_name in attribution_results.keys():
    for i in range(min(3, len(sample_indices))):
        print(f'\nDetailed comparison: {model_name} - sample {i+1}...')
        compare_attribution_methods(model_name, i)

print('\n' + '='*60)
print('All visualizations complete!')
print(f'Saved to: {OUTPUT_DIR.absolute()}')
print('='*60)

In [ ]:
# Summary statistics
print('\n' + '='*80)
print('ATTRIBUTION ANALYSIS SUMMARY')
print('='*80)

for model_name, results in attribution_results.items():
    print(f'\n{model_name}:')
    print('-' * 60)
    
    # Prediction accuracy
    predictions = results['predictions']
    correct = sum(1 for p in predictions if p['correct'])
    accuracy = correct / len(predictions)
    print(f'  Accuracy: {correct}/{len(predictions)} ({accuracy:.2%})')
    
    # Attribution method success rates
    print(f'\n  Attribution Method Success Rates:')
    for method_name in ATTRIBUTION_METHODS.keys():
        attributions = results['attributions'][method_name]
        successful = sum(1 for a in attributions if a is not None)
        rate = successful / len(attributions)
        print(f'    {method_name:<20}: {successful}/{len(attributions)} ({rate:.2%})')

print('\n' + '='*80)
print(f'Total visualizations saved: {len(list(OUTPUT_DIR.glob("*.png")))}')
print(f'Output directory: {OUTPUT_DIR.absolute()}')
print('='*80)

## Next Steps

### Analysis Options:

1. **Visualize specific samples:**

   ```python
   visualize_sample_attributions(sample_idx=2)
   ```

2. **Visualize specific model:**

   ```python
   visualize_model_attributions('resnet18')
   ```

3. **Compare methods:**
   ```python
   compare_attribution_methods('resnet18', sample_idx=0)
   ```

### Interpretation:

- **Red regions**: Areas the model focuses on for its prediction
- **Blue regions**: Less important areas
- **Compare across methods**: Do different methods agree on important regions?
- **Compare across models**: Do different architectures focus on the same features?

### Export Results:

All visualizations are saved to `outputs/attributions_viz/`
